# Biopython으로 인플루엔자 서열 분석해보기
## 개요
여러분. 이번에는 H5N1입니다.

얘는 철새, 닭, 오리 등의 새가 숙주인데 사람한테도 감염되는 인수공통 감영병입니다. 사람과 사람 사이에 전염되는건 환자를 격리하던가 마스크를 씌운다던가 하면 막을 수 있지만, 이거는 아... 철새를 어떻게 막습니까... 막말로 다 쏴죽일 것도 아니고...

## 프로젝트 정보
- 인원: 1인(개인 프로젝트)
- 버전: 3.10(TF_base)
- 설치할 것들: Biopython, muscle
- 데이터 리소스: NCBI(Entrez로 갖고올 예정)

In [ ]:
# 모듈
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# BioPython
from Bio import Entrez, SeqIO # 왼쪽: 일단 털어보자/오른쪽: 시퀀스 다루려면 필요합니다. 필수임.
from Bio import AlignIO # 서열 분석해줄 친구
from Bio import Phylo # 트리 그릴라면 필요해요
from Bio.Phylo.TreeConstruction import DistanceCalculator, DistanceTreeConstructor
from Bio.Align import MultipleSeqAlignment
from Bio.Seq import Seq

# 통계분석용
from scipy.stats import mannwhitneyu
from itertools import combinations
from scipy.stats import spearmanr
from sklearn.linear_model import LinearRegression # 잔차분석용

import io # 누구세요?
import itertools
import subprocess # 서브 프로세스(이건 또 뭐여...)
from collections import Counter
import re
import math

In [ ]:
# 그래프를 그리기 위한 기본 설정
plt.rcParams['font.family'] = 'Nanumsquare_ac' # 나눔바른펜(본인 기본 고딕 싫어함)
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['font.size'] = 14
plt.rcParams['axes.unicode_minus'] = False

# 사전세팅
Entrez.email = "blackholekun@gmail.com" # 이메일
muscle_exe = "/opt/homebrew/bin/muscle" # 이거 경로 있어야 써요(which 치면 나옴)

# 여러분, 창고는 언제나 열려있을 것입니다.
- 근데 이번에는 H3N2가 아니고 H5N1을 찾아야 합니다.

In [ ]:
# 인플루엔자 H3N2 서열 가져오기
# 쿼리 조건: 인플루엔자 A, 특정 아형, HA 유전자, PDAT(1년)
# 얘는 숙주를 사람으로 제한하면 새랑 사람간의 차이를 볼 수 없습니다.
query = f"Influenza A virus AND H5N1 AND HA[Gene Name] AND 2025[PDAT] AND 2024/02/05:2026/02/05[PDAT]"

# 1. ID 리스트 가져오기
handle = Entrez.esearch(db="nucleotide", term=query, retmax=300)
record = Entrez.read(handle)
id_list = record["IdList"]
handle.close()

# 2. 실제 서열 데이터 가져오기 (FASTA 형식)
fetch_handle = Entrez.efetch(db="nucleotide", id=id_list, rettype="fasta", retmode="text")
sequences = list(SeqIO.parse(fetch_handle, "fasta"))
fetch_handle.close()

# 3. 저장
with open("influenza_h5n1.fasta", "w") as f:
    SeqIO.write(sequences, f, "fasta")

print(f"성공적으로 {len(sequences)}개의 서열을 가져왔습니다.")
print("----------")

for record in sequences[:3]:
    print(f"ID: {record.id}")
    print(f"Description: {record.description}")
    print(f"Length: {len(record.seq)} bp\n")

## 필터링 & 정보 확인

In [ ]:
# 1. 먼저 H5N1 서열만 필터링해서 따로 모읍니다.
h5n1_only = [
    record for record in sequences
    if "H5N1" in record.id or "H5N1" in record.description
]

# 2. 필터링된 서열들 중에서만 최소 길이를 찾습니다.
# (전체 sequences 기준이 아니라 h3n2_only 기준으로 해야 정확합니다)
min_len = min(len(s.seq) for s in h5n1_only)
print(f"H5N1 서열 개수: {len(h5n1_only)}")
print(f"맞춤 길이: {min_len} bp")

# 3. [중요] 필터링된 서열들을 자른 '새로운 리스트'를 만듭니다.
trimmed_h5n1 = []
for record in h5n1_only:
    trimmed_h5n1.append(record[:min_len])

# 4. [핵심] 이제 '자른 리스트'인 trimmed_h3n2를 넣어야 에러가 안 납니다!
alignment = MultipleSeqAlignment(trimmed_h5n1)

print(f"MultipleSeqAlignment 생성 성공! 서열 수: {len(alignment)}")

In [ ]:
# 사전통계-어느 지역 데이터를 얼마나 긁어왔는가?
locations = []
for record in sequences:
    # Description에서 괄호 안의 지역 정보 추출 (예: A/Shanghai/...)
    match = re.search(r'A/([^/]+)/', record.description)
    if match:
        locations.append(match.group(1))

# 지역별 빈도수 확인
location_counts = Counter(locations)
print("--- 수집된 데이터 지역 분포 ---")
for loc, count in location_counts.most_common():
    print(f"{loc}: {count}개")

- 어째 다 새다...
- Cattle: 소
- Domestic cat: 냥이

In [ ]:
# 이런 식으로 그룹화해서 분석하면 결과가 훨씬 명확해집니다.
host_map = {
    'cattle': 'Mammal',
    'Domestic Cat': 'Mammal',
    'Turkey': 'Avian',
    'chicken': 'Avian',
    'Duck': 'Avian',
    'Domestic Duck': 'Avian',
    'northern pintail': 'Avian',
    # ... 나머지도 Avian으로 통합
}

# MSA

In [ ]:
print('MSA start... ')

# MSA 분석 시-작
try:
    result = subprocess.run([muscle_exe, "-align", "influenza_h5n1.fasta", "-output", "influenza_h5n1_muscle_aligned.fasta"], check=True, capture_output=True, text=True)
    print("Completed. ")
except subprocess.CalledProcessError as e:
    print(f"MSA failed: {e}")
finally:
    alignment = AlignIO.read("influenza_h5n1_muscle_aligned.fasta", "fasta")

# 오래 걸리니까 이거 돌려놓고 잠깐 바람 쐬고 오십쇼

In [ ]:
print("====== MSA Result ======")
alignment = AlignIO.read("influenza_h5n1_muscle_aligned.fasta", "fasta") # FASTA 니네 확장자가 몇개냐...

for record in alignment:
    print(f"{record.id[:15]:<15} : {record.seq[:110]}")

# 섀넌 엔트로피 분석

## 평균 보존율

In [ ]:
def calculate_conservation_no_gap(alignment, gap_threshold=0.5):
    length = alignment.get_alignment_length()
    scores = []

    for i in range(length):
        column_raw = alignment[:, i]

        # gap 비율이 너무 높으면 제외 (선택사항)
        gap_fraction = column_raw.count("-") / len(column_raw)
        if gap_fraction > gap_threshold:
            continue

        # gap 제거
        column = column_raw.replace("-", "")
        if len(column) == 0:
            continue

        # 최빈 염기 비율 = 보존도
        most_common = max(set(column), key=column.count)
        score = column.count(most_common) / len(column)
        scores.append(score)

    return scores

scores = calculate_conservation_no_gap(alignment)

print(f"해당 구간의 평균 보존율: {np.mean(scores)*100:.2f}%")

## 변이 핫스팟

In [ ]:
# 섀넌 엔트로피 점수 도출
def get_top_variable_sites_no_gap(alignment, top_n=10):
    length = alignment.get_alignment_length()
    variability = []

    ref_seq = alignment[0].seq

    for i in range(length):
        # 🔴 reference가 gap이면 무조건 스킵
        if ref_seq[i] == '-':
            continue

        column = alignment[:, i].replace("-", "")
        if not column:
            continue

        counts = Counter(column)
        total = sum(counts.values())

        entropy = 0.0
        for c in counts.values():
            p = c / total
            entropy -= p * math.log2(p)

        variability.append((i, entropy))

    return sorted(variability, key=lambda x: x[1], reverse=True)[:top_n]

def alignment_to_sequence_pos(aligned_seq, aln_pos):
    count = 0
    for i in range(aln_pos + 1):
        if aligned_seq[i] != '-':
            count += 1
    return count


ref_seq = alignment[0].seq
top_sites = get_top_variable_sites_no_gap(alignment, top_n=10)

high_entropy_ha_sites = []

print("--- 변이가 집중된 주요 포지션 분석 결과 ---")
for aln_pos, score in top_sites:
    real_pos = alignment_to_sequence_pos(ref_seq, aln_pos)
    high_entropy_ha_sites.append(real_pos)
    print(f"Alignment {aln_pos:4d} → Pos {real_pos:4d} | 엔트로피: {score:.3f}")

print("\n최종 고엔트로피 포지션 리스트:")
print(high_entropy_ha_sites)

## 섀넌 엔트로피 분석

In [ ]:
def shannon_entropy_no_gap(alignment):
    length = alignment.get_alignment_length()
    ref_seq = alignment[0].seq

    entropy_scores = []

    for i in range(length):
        # reference가 gap이면 제외
        if ref_seq[i] == '-':
            entropy_scores.append(np.nan)
            continue

        column = alignment[:, i].replace("-", "")
        if not column:
            entropy_scores.append(np.nan)
            continue

        counts = Counter(column)
        total = sum(counts.values())

        entropy = 0.0
        for c in counts.values():
            p = c / total
            entropy -= p * math.log2(p)

        entropy_scores.append(entropy)

    return np.array(entropy_scores)

def sliding_window_mean(values, window=20):
    """
    values : np.array (entropy scores, np.nan 포함)
    window : window size
    """
    smoothed = []

    for i in range(len(values)):
        start = max(0, i - window // 2)
        end = min(len(values), i + window // 2 + 1)

        window_vals = values[start:end]
        window_vals = window_vals[~np.isnan(window_vals)]

        if len(window_vals) == 0:
            smoothed.append(np.nan)
        else:
            smoothed.append(np.mean(window_vals))

    return np.array(smoothed)

In [ ]:
entropy_raw = shannon_entropy_no_gap(alignment)
entropy_window = sliding_window_mean(entropy_raw, window=25)

In [ ]:
plt.figure(figsize=(15, 5))

plt.plot(entropy_raw, color='#009698', linewidth=2)
plt.fill_between(
    range(len(entropy_raw)),
    entropy_raw,
    color='skyblue',
    alpha=0.25
)

plt.axvline(1074, linestyle='--', color='red', alpha=0.6)
plt.text(1074, 1.07, 'POS 1074', color='red', alpha=0.5, ha='center', fontweight='bold')

plt.axvline(960, linestyle='--', color='red', alpha=0.6)
plt.text(960, 1.12, 'POS 960', color='red', alpha=0.5, ha='center', fontweight='bold')

plt.axvline(327, linestyle='--', color='red', alpha=0.6)
plt.text(327, 1.07, 'POS 327', color='red', alpha=0.5, ha='center', fontweight='bold')

plt.title("Viral Variation Hotspots (Gap-filtered Shannon Entropy)", y = 1.12)
plt.xlabel("Alignment Position (filtered)")
plt.ylabel("Normalized Shannon Entropy")
plt.show()

### 통계분석 (섀넌 엔트로피)
- 귀무가설: 인플루엔자 H5N1의 해마글루티닌 변이는 무작위적으로 발생하며, 특정 위치에 선호적으로 집중되지 않는다.
- 대립가설: 인플루엔자 H5N1의 해마글루티닌 변이는 무작위가 아니며, 특정 위치(hotspots)에 유의하게 집중된다.

In [ ]:
variation_scores = entropy_raw[~np.isnan(entropy_raw)]

mean_var = np.mean(variation_scores)
median_var = np.median(variation_scores)
iqr_var = np.percentile(variation_scores, 75) - np.percentile(variation_scores, 25)

print(f"Mean variation score: {mean_var:.4f}")
print(f"Median variation score: {median_var:.4f}")
print(f"IQR: {iqr_var:.4f}")

In [ ]:
# Define high-variation hotspots (top 10%)
threshold = np.percentile(variation_scores, 90)

hotspots = variation_scores[variation_scores >= threshold]
non_hotspots = variation_scores[variation_scores < threshold]

u_stat, p_value = mannwhitneyu(
    hotspots,
    non_hotspots,
    alternative="greater"
)

print(f"Hotspot threshold (90th percentile): {threshold:.4f}")
print(f"Mann–Whitney U statistic: {u_stat:.1f}")
print(f"p-value: {p_value:.4e}" if p_value > 1e-10 else "p-value: <1e-10")

<속보> 귀무가설 압도적 기각

# Phylogenic tree

In [ ]:
# 트! 리!
calculator = DistanceCalculator('identity')
dm = calculator.get_distance(alignment)
constructor = DistanceTreeConstructor(calculator, 'nj')
tree = constructor.build_tree(alignment)

terms = tree.get_terminals()
x_limit = max([tree.distance(t) for t in terms])
fig = plt.figure(figsize=(20, 60), dpi=150) # 난 해상도 설정도 될 줄 몰랐고...
ax = fig.add_subplot(1, 1, 1)

for clade in tree.get_terminals():
    original_name = str(clade.name)
    if '_' in original_name:
        parts = original_name.split('_')
        clade.name = f"[{parts[0]}] {parts[1]} ({original_name})"
    else:
        clade.name = original_name

Phylo.draw(tree, axes=ax, do_show=False, label_func=lambda x: "", show_confidence=False)

# 내가 진짜 이것때문에 제미나이랑 급나 씨름했는데 색깔이 안바껴요.
# 이름도 몇번이나 했는데 ID만 줄창떠요. 아오.
for i, node in enumerate(terms):
    y_pos = i + 1  # 가지의 y축 위치
    x_pos = tree.distance(node) # 가지가 끝나는 x축 위치

    orig_name = str(node.name)
    # 이름 가공: [연도] 지역 (ID)
    if '_' in orig_name:
        p = orig_name.split('_')
        # 혹시 이미 가공된 이름이라면 중복 방지
        display_text = f"  ◀ [{p[0]}] {p[1]}" if '[' not in orig_name else f"  ◀ {orig_name}"
    else:
        display_text = f"  ◀ {orig_name}"

    # 가지 끝(x_pos)에 바로 텍스트를 박습니다.
    ax.text(x_pos, y_pos, display_text,
            va='center', ha='left',
            fontsize=14,
            fontweight='bold' if "LC909067" in orig_name else 'normal')

ax.set_xlim(0, x_limit * 1.8)
ax.set_ylim(0, len(terms) + 2)
ax.set_axis_off() # 축 숫자 빠잉

plt.rc('font', size=14) # 내부 글꼴 사이즈
plt.rc('axes', titlesize=20) # 제모옥은 이 크기로 하겠습니다
plt.title("Influenza A (H3N2) HA Phylogenetic Tree by Region/Year")
plt.tight_layout()
plt.savefig("Influenza_H3N2_Final_Tree.png", dpi=300, bbox_inches='tight')
plt.xlabel("Genetic Distance (Substitutions per site)")
plt.show()

- 그... 분리가 거의 안된거 아닌가...

## 통계분석 (스피어만 상관계수)
- 귀무가설: 트리 거리와 서열 유사도는 상관이 없다 → 계통수 구조는 서열 차이를 반영하지 않는다.
- 대립가설: 트리 거리와 서열 유사도간에 서로 상관이 있다 → 계통수 구조는 서열 차이를 반영했다.

In [ ]:
tree_distances = []
seq_identities = []

terms = tree.get_terminals()

def pairwise_identity(seq1, seq2):
    # 두 서열 중 어느 한쪽이라도 갭이 아닌 위치만 골라냄
    matches = 0
    total_valid_length = 0
    for s1, s2 in zip(seq1, seq2):
        if s1 == '-' and s2 == '-': # 둘 다 갭이면 무시
            continue
        total_valid_length += 1
        if s1 == s2:
            matches += 1

    return (matches / total_valid_length) if total_valid_length > 0 else 0

for rec1, rec2 in combinations(alignment, 2):
    id1 = rec1.id.split('.')[0]
    id2 = rec2.id.split('.')[0]

    try:
        # 가공된 트리 이름 속에서 원본 ID가 포함된 노드를 각각 찾음
        node1 = [t for t in terms if id1 in t.name][0]
        node2 = [t for t in terms if id2 in t.name][0]

        d = tree.distance(node1, node2)
        iden = pairwise_identity(str(rec1.seq), str(rec2.seq))

        tree_distances.append(d)
        seq_identities.append(iden)
    except IndexError:
        # 트리에 해당 ID가 없는 경우 건너뜀
        continue

In [ ]:
rho, p = spearmanr(tree_distances, seq_identities)

print(f'Rho: {rho:.4f}')
print(f'P-value: {p:.4e}')

## 시각화

In [ ]:
plt.figure(figsize=(10, 7), dpi=120)
sns.regplot(x=tree_distances, y=seq_identities,
            scatter_kws={'alpha':0.2, 'color':'gray', 's':10},
            line_kws={'color':'red', 'label': f'Spearman Rho: {rho:.3f}'})

plt.title("H5N1 HA: Tree Distance vs Sequence Identity", fontsize=15, pad=15)
plt.xlabel("Genetic Distance on Tree", fontsize=12)
plt.ylabel("Pairwise Sequence Identity", fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## 잔차 분석
- 솔직히 저기 떨어져있는 점들 궁금하시죠? 저는 궁금합니다.

In [ ]:
# 1. 데이터 준비 (x: Tree Distance, y: Sequence Identity)
X = np.array(tree_distances).reshape(-1, 1)
y = np.array(seq_identities)

# 2. 선형 회귀 모델 생성 및 학습
model = LinearRegression()
model.fit(X, y) # 아 단항인가

# 3. 예측값 계산 및 잔차(Residual) 산출
predictions = model.predict(X)
residuals = y - predictions  # 실제값 - 예측값

# 4. 잔차가 큰 상위 10개 인덱스 추출 (절댓값 기준)
outlier_indices = np.argsort(np.abs(residuals))[-10:]

# 절댓값이 아니라, 그냥 residuals 값이 가장 작은(가장 큰 음수) 순서대로 정렬
fast_outlier_indices = np.argsort(residuals)[:10]

print(f"{'Rank':<5} | {'Pair Index':<12} | {'Residual':<10} | {'Note'}")
print("-" * 50)
for i, idx in enumerate(fast_outlier_indices, 1):
    res_val = residuals[idx]
    print(f"{i:<5} | {idx:<12} | {res_val:<10.4f} | Faster Evolution 🔥")

In [ ]:
# 1. 원래 alignment에 넣었던 서열 리스트의 길이를 n이라고 하면
n = len(alignment)

# 2. combinations를 리스트로 변환해서 44094번 인덱스를 직접 찾습니다.
pairs = list(itertools.combinations(range(n), 2))
idx1, idx2 = pairs[44094]

# 3. 드디어 범인들의 이름(Description)을 출력합니다!
print(f"--- 44094번 페어의 정체 ---")
print(f"서열 A (Host: {alignment[idx1].annotations.get('host', 'Unknown')}): {alignment[idx1].description}")
print(f"서열 B (Host: {alignment[idx2].annotations.get('host', 'Unknown')}): {alignment[idx2].description}")

- 회귀분석이 뭔지는 아시죠? 독립 변수(원인)가 종속 변수(결과)에 미치는 영향의 패턴을 수학적으로 모델링하고 예측하는 통계적 방법입니다. 회귀식... 쟈는 1차같은데...
- 잔차가 뭐냐... 회귀식이 직선입니다. 그리고 거기서 실제 값이 살짝 벗어난 경우가 있잖아요. 그 실제 값이랑 모델이 예측한 값이랑 차이가 잔차입니다. 이건 회귀분석을 통해 그 삐져나온 놈들이 누구냐! 를 본 거예요.
- 근데 잔차 같은건 다 소랑 핀테일(새)같은데...

# 이 중에 변이왕은 누구인가

In [ ]:
# 사람을 공격하는 바이러스만 찾습니다
human_terms = [t for t in tree.get_terminals() if 'canine' not in str(t.name).lower()]

# 유전적 거리순으로 정렬
human_distances = [(tree.distance(t), t.name) for t in human_terms]
human_distances.sort(key=lambda x: x[0], reverse=True)

print("=== 🚨 독감 변종 TOP 5 ===")
print("-" * 70)
print(f"{'순위':<4} | {'ID':<12} | {'변이도':<8} | {'신상 정보'}")
print("-" * 70)

for i, (dist, name) in enumerate(human_distances[:5], 1):
    target_id = str(name)
    found_info = "정보 없음"

    # alignment 데이터에서 상세 지역/연도 정보 매칭
    for record in alignment:
        if target_id in record.description or target_id in record.id:
            full_info = record.description if record.description else record.id
            if '_' in full_info:
                parts = full_info.split('_')
                found_info = f"[{parts[0]}] {parts[1].split(' ')[0]}"
            else:
                # description에서 연도/지역 추출 시도 (괄호 안 정보 등)
                found_info = full_info.split('virus (')[1].split(')')[0] if '(' in full_info else full_info
            break

    print(f"{i:<5} | {target_id:<12} | {dist:.4f} | {found_info}")

print("-" * 70)
print("※ 변이도가 높을수록 기존 면역 체계를 회피할 가능성이 클 수도 있습니다.")